In [9]:
import pandas as pd
import numpy as np
from pathlib import Path
import sqlite3 as sql
from dotenv import load_dotenv, dotenv_values
import os

load_dotenv(Path.cwd().parent / ".env")

True

In [2]:
data = pd.read_csv(f'{Path.cwd().parent / "com_out" / "companies_house_data.csv"}')

C:\Users\Ink\AppData\Local\Temp\ipykernel_4484\1795661269.py:1: DtypeWarning: Columns (0: RegAddress.POBox, 1: PreviousName_5.CONDATE, 2:  PreviousName_5.CompanyName, 3: PreviousName_6.CONDATE, 4:  PreviousName_6.CompanyName, 5: PreviousName_7.CONDATE, 6:  PreviousName_7.CompanyName, 7: PreviousName_8.CONDATE, 8:  PreviousName_8.CompanyName, 9: PreviousName_9.CONDATE, 10:  PreviousName_9.CompanyName, 11: PreviousName_10.CONDATE, 12:  PreviousName_10.CompanyName) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(f'{Path.cwd().parent / "com_out" / "companies_house_data.csv"}')


In [3]:
len(data)

5698274

In [4]:
len(data.columns)

55

In [6]:
SMEs = ["SMALL", 'MICRO ENTITY', 'TOTAL EXEMPTION FULL', 'TOTAL EXEMPTION SMALL', "MEDIUM"]

active = data.loc[data["CompanyStatus"] == "Active"]
active = active.loc[active["Accounts.AccountCategory"] != "DORMANT"]
active = active.loc[active["CompanyName"].str.contains("LTD|LIMITED|LLP|LIMITED LIABILITY PARTNERSHIP", case=False, na=False)]
# active = active.loc[active["Accounts.AccountCategory"].isin(SMEs)]  
active.rename(columns={" CompanyNumber": "CompanyNumber", "IncorporationDate": "DateOfCreation","RegAddress.PostTown":"Town", "RegAddress.County":"County", "RegAddress.PostCode": "PostCode"}, inplace=True)


In [7]:
len(active)

4390812

In [ ]:
active["Accounts.AccountCategory"].unique()

<StringArray>
[          'NO ACCOUNTS FILED',        'TOTAL EXEMPTION FULL',
                'MICRO ENTITY',                        'FULL',
          'UNAUDITED ABRIDGED',                       'GROUP',
                       'SMALL',  'AUDIT EXEMPTION SUBSIDIARY',
                      'MEDIUM', 'ACCOUNTS TYPE NOT AVAILABLE',
       'TOTAL EXEMPTION SMALL', 'FILING EXEMPTION SUBSIDIARY',
            'AUDITED ABRIDGED',           'PARTIAL EXEMPTION']
Length: 14, dtype: str

In [ ]:
active["CompanyNumber"]

1          16092999
3          16873705
4          15073164
5          13522064
6          11006939
             ...   
5698264    14418099
5698267    14079291
5698268    11044986
5698271    09511422
5698272    11457383
Name: CompanyNumber, Length: 4390812, dtype: str

In [46]:
active.iloc[100]["Accounts.AccountCategory"]

'TOTAL EXEMPTION FULL'

In [47]:
active_names = active[["CompanyName", "CompanyNumber","DateOfCreation","CompanyCategory", "Town","County","PostCode","SICCode.SicText_1","SICCode.SicText_2","SICCode.SicText_3","SICCode.SicText_4"]]

In [51]:
active_names

,CompanyName,CompanyNumber,DateOfCreation,CompanyCategory,Town,County,PostCode,SICCode.SicText_1,SICCode.SicText_2,SICCode.SicText_3,SICCode.SicText_4
4,!NFLECTION ADVISORY LIMITED,15073164,15/08/2023,Private Limited Company,POTTERS BAR,HERTFORDSHIRE,EN6 2DA,70229 - Management consultancy activities othe...,NaN,NaN,NaN
5,!NFOGENIE LTD,13522064,21/07/2021,Private Limited Company,LONDON,GREATER LONDON,WC2H 9JQ,58290 - Other software publishing,NaN,NaN,NaN
6,!NNOV8 LIMITED,11006939,11/10/2017,Private Limited Company,EDENBRIDGE,NaN,TN8 5NF,62090 - Other information technology service a...,70229 - Management consultancy activities othe...,NaN,NaN
7,!NSPIRED INVESTMENTS LTD,SC606050,22/08/2018,Private Limited Company,ABERDEEN,NaN,AB11 7SY,68209 - Other letting and operating of own or ...,NaN,NaN,NaN
12,"""1ST RATE"" PSYCHOLOGY SERVICES LTD",11303802,11/04/2018,Private Limited Company,GREAT WAKERING,ESSEX,SS3 0GW,85600 - Educational support services,86900 - Other human health activities,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
5698264,‘HEOL CHWARAE RÔL LTD,14418099,13/10/2022,"PRI/LTD BY GUAR/NSC (Private, limited by guara...",PONTYPRIDD,NaN,CF38 1TQ,93290 - Other amusement and recreation activit...,NaN,NaN,NaN
5698267,"“34 HALLAM ROAD FLAT MANAGEMENT COMPANY"" LIMITED",14079291,29/04/2022,Private Limited Company,CLEVEDON,NaN,BS21 7NE,99999 - Dormant Company,NaN,NaN,NaN
5698268,“ARTIS UK” LIMITED,11044986,02/11/2017,Private Limited Company,SURBITON,NaN,KT6 5LX,41202 - Construction of domestic buildings,NaN,NaN,NaN
5698271,“SAIL IN GREECE ADVENTURES” LTD,09511422,26/03/2015,Private Limited Company,LONDON,LONDON,EC3V 3NG,79120 - Tour operator activities,NaN,NaN,NaN


In [52]:
def add_company_names_to_db(df):
    con = sql.connect(Path.cwd().parent / os.getenv("DB_URL"))
    cur = con.cursor()

    for index, row in df.iterrows():
        cur.execute("INSERT INTO companies (name, company_id, date_of_creation, company_type, town, county, post_code) VALUES (?, ?, ?, ?, ?, ?, ?)", (row["CompanyName"], row["CompanyNumber"], row["DateOfCreation"], row["CompanyCategory"], row["Town"], row["County"], row["PostCode"]))
    con.commit()
    con.close()
    
def add_sic_codes_to_db(df):
    con = sql.connect(Path.cwd().parent / os.getenv("DB_URL"))
    cur = con.cursor()

    for index, row in df.iterrows():
        sic_codes = row[["SICCode.SicText_1", "SICCode.SicText_2", "SICCode.SicText_3", "SICCode.SicText_4"]].dropna().tolist()
        if 'None Supplied' in sic_codes:
            sic_codes.remove('None Supplied')
        for i in sic_codes:
            code, text = i.split(" - ")
            try:
                cur.execute("INSERT INTO sic_codes (sic_id, sector_name) VALUES (?, ?)", (code.strip(), text.strip()))
            except sql.IntegrityError:
                continue
    
    con.commit()
    con.close()        
    
def add_company_sic_codes_to_db(df):
    con = sql.connect(Path.cwd().parent / os.getenv("DB_URL"))
    cur = con.cursor()

    for index, row in df.iterrows():
        company_id = row["CompanyNumber"]
        sic_codes = row[["SICCode.SicText_1", "SICCode.SicText_2", "SICCode.SicText_3", "SICCode.SicText_4"]].dropna().tolist()
        if 'None Supplied' in sic_codes:
            sic_codes.remove('None Supplied')
        for i in sic_codes:
            code, text = i.split(" - ")
            try:
                cur.execute("INSERT INTO companies_sic_codes (company_id, sic_code) VALUES (?, ?)", (company_id, code.strip()))

            except sql.IntegrityError:
                continue

    con.commit()
    con.close()

In [53]:
add_company_names_to_db(active_names)

In [54]:
add_sic_codes_to_db(active_names)

In [55]:
add_company_sic_codes_to_db(active_names)

In [18]:
result = []
conn = sql.connect(Path.cwd().parent / os.getenv("DB_URL"))
cursor = conn.cursor()

cursor.execute("SELECT c.company_id, c.name, c.date_of_creation, c.company_type, town, county, post_code, sic_code  FROM companies c JOIN companies_sic_codes  csc ON c.company_id = csc.company_id")
out = cursor.fetchall()

out

[('15073164',
  '!NFLECTION ADVISORY LIMITED',
  '15/08/2023',
  'Private Limited Company',
  'POTTERS BAR',
  'HERTFORDSHIRE',
  'EN6 2DA',
  '70229'),
 ('13522064',
  '!NFOGENIE LTD',
  '21/07/2021',
  'Private Limited Company',
  'LONDON',
  'GREATER LONDON',
  'WC2H 9JQ',
  '58290'),
 ('11006939',
  '!NNOV8 LIMITED',
  '11/10/2017',
  'Private Limited Company',
  'EDENBRIDGE',
  None,
  'TN8 5NF',
  '62090'),
 ('11006939',
  '!NNOV8 LIMITED',
  '11/10/2017',
  'Private Limited Company',
  'EDENBRIDGE',
  None,
  'TN8 5NF',
  '70229'),
 ('SC606050',
  '!NSPIRED INVESTMENTS LTD',
  '22/08/2018',
  'Private Limited Company',
  'ABERDEEN',
  None,
  'AB11 7SY',
  '68209'),
 ('11303802',
  '"1ST RATE" PSYCHOLOGY SERVICES LTD',
  '11/04/2018',
  'Private Limited Company',
  'GREAT WAKERING',
  'ESSEX',
  'SS3 0GW',
  '85600'),
 ('11303802',
  '"1ST RATE" PSYCHOLOGY SERVICES LTD',
  '11/04/2018',
  'Private Limited Company',
  'GREAT WAKERING',
  'ESSEX',
  'SS3 0GW',
  '86900'),
 ('05914

In [19]:
data = pd.DataFrame(out, columns=["company_id", "com_name", "date_of_creation", "com_type", "town", "county", "post_code", "sic_code"])

In [20]:
company_ids = list(data["company_id"].unique())

In [21]:
len(company_ids)

2864268

In [22]:
import random

random_company_ids = random.sample(company_ids, 30000)

In [28]:
final = data[data["company_id"].isin(random_company_ids)]

In [29]:
final.to_csv(Path.cwd().parent / "com_out" / "30k_sme_for_mock_up.csv", index=False)